# 02_kassel_exploration.ipynb

## Σκοπός

Αυτό το notebook είναι το **strict raw validation gate** της ενεργής DaKS / Kassel forecasting pipeline.

Ελέγχει αποκλειστικά:

- strict input-target pairing ανά `park_id`
- recursive raw file discovery κάτω από το `data/raw/`
- separator resolution ανά αρχείο
- strict timestamp parsing με **explicit allowed formats**
- duplicate timestamp detection
- ordering audit πριν από canonical sorting
- exact input-target timestamp alignment
- one-to-one merge validity
- park-level audit artifact generation για downstream χρήση

## Τι δεν κάνει

Αυτό το notebook **δεν** είναι:

- EDA notebook
- cleaning notebook
- feature engineering notebook
- split notebook
- modeling notebook

## Methodological contract

Το `NB02` είναι η **μοναδική raw parsing authority** του pipeline.

Αυτό σημαίνει ότι:

- το canonical parsed timestamp ορίζεται εδώ
- οποιαδήποτε αποτυχία parsing / alignment καταγράφεται εδώ
- το `NB03` δεν επιτρέπεται να βασιστεί σε loose reparsing από raw strings
- μόνο parks που περνούν το strict audit μπορούν να συνεχίσουν downstream

In [1]:
# ---------------------------------------------------------------------
# Imports και path resolution
# ---------------------------------------------------------------------
import re
from pathlib import Path

import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm

# Ρυθμίσεις εμφάνισης για καθαρότερα audit tables.
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)


def find_project_root(start_path: Path) -> Path:
    """
    Εντοπίζει το project root ανεβαίνοντας ιεραρχικά από το current working
    directory μέχρι να βρει το βασικό repository layout.

    Αναμένουμε τουλάχιστον:
    - data/
    - notebooks/
    """
    current = start_path.resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate

    raise FileNotFoundError(
        "Δεν βρέθηκε project root με φακέλους 'data/' και 'notebooks/'. "
        "Άνοιξε το notebook μέσα από το repository root ή από τον φάκελο notebooks/."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

# Το raw root είναι το γενικό container του dataset.
RAW_ROOT = PROJECT_ROOT / "data" / "raw"

# Το repo documentation δείχνει ως canonical setup το:
# data/raw/kassel_dataset/
# Όμως για robustness δεν θα στηριχθούμε μόνο εκεί· θα κάνουμε recursive scan
# κάτω από όλο το RAW_ROOT.
PREFERRED_RAW_DATASET_DIR = RAW_ROOT / "kassel_dataset"

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

if not RAW_ROOT.exists():
    raise FileNotFoundError(
        f"Δεν βρέθηκε ο φάκελος raw data: {RAW_ROOT}\n"
        "Το NB02 απαιτεί το DaKS raw dataset να υπάρχει ήδη τοπικά."
    )

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


def to_raw_relative(path: Path | None) -> str | None:
    """
    Μετατρέπει ένα απόλυτο path σε path σχετικό με το RAW_ROOT για πιο καθαρά logs.
    """
    if path is None:
        return None

    return str(path.resolve().relative_to(RAW_ROOT.resolve()))


print(f"PROJECT_ROOT               : {PROJECT_ROOT}")
print(f"RAW_ROOT                   : {RAW_ROOT}")
print(f"PREFERRED_RAW_DATASET_DIR  : {PREFERRED_RAW_DATASET_DIR}")
print(f"PROCESSED_DIR              : {PROCESSED_DIR}")

PROJECT_ROOT               : C:\Users\diony\Desktop\WindPower_DigitalTwin
RAW_ROOT                   : C:\Users\diony\Desktop\WindPower_DigitalTwin\data\raw
PREFERRED_RAW_DATASET_DIR  : C:\Users\diony\Desktop\WindPower_DigitalTwin\data\raw\kassel_dataset
PROCESSED_DIR              : C:\Users\diony\Desktop\WindPower_DigitalTwin\data\processed


## Strict pairing policy

Το raw DaKS dataset βασίζεται σε park-level αρχεία της μορφής:

- `data_input_<park_id>.csv`
- `data_target_<park_id>.csv`

Επιπλέον, αναμένεται auxiliary metadata file:

- `meta.csv`

Η πρώτη κρίσιμη συνθήκη είναι το **strict one-input / one-target pairing** ανά `park_id`.

Σε αυτό το στάδιο:

- δεν επιτρέπεται loose matching
- δεν επιτρέπεται ad-hoc file selection
- δεν επιτρέπεται ambiguity λόγω duplicate pair files σε διαφορετικούς υποφακέλους
- το notebook πρέπει να υποστηρίζει nested raw layout κάτω από `data/raw/`

Άρα το file discovery γίνεται **recursive** κάτω από το `data/raw/`, αλλά το pairing παραμένει strict και fail-fast.

In [2]:
# ---------------------------------------------------------------------
# Recursive file discovery και strict pairing audit
# ---------------------------------------------------------------------

# Επιτρέπονται μόνο raw park-level αρχεία αυτού του pattern.
PAIR_PATTERN = re.compile(r"^data_(input|target)_(\d+)\.csv$", re.IGNORECASE)

# Κάνουμε recursive scan κάτω από όλο το data/raw ώστε να καλύπτεται και το
# documented layout data/raw/kassel_dataset/ αλλά και τυχόν ισοδύναμα nested layouts.
all_csv_files = sorted(RAW_ROOT.rglob("*.csv"))

if not all_csv_files:
    raise RuntimeError(
        f"Δεν βρέθηκαν CSV αρχεία κάτω από το raw root: {RAW_ROOT}"
    )

input_files: dict[str, Path] = {}
target_files: dict[str, Path] = {}

# Auxiliary raw files όπως το meta.csv δεν είναι pair files και δεν πρέπει να
# θεωρούνται σφάλμα pairing.
meta_file_path: Path | None = None
auxiliary_csv_files: list[str] = []

# Αν βρεθούν δύο input ή δύο target files για το ίδιο park_id σε διαφορετικά paths,
# αυτό δημιουργεί ambiguity και πρέπει να αποτύχει ρητά.
duplicate_pair_rows: list[dict] = []

for path in all_csv_files:
    rel_path = to_raw_relative(path)
    file_name = path.name

    # Το meta.csv είναι expected auxiliary file.
    if file_name.lower() == "meta.csv":
        if meta_file_path is not None:
            raise RuntimeError(
                "Βρέθηκαν πολλαπλά meta.csv files κάτω από το data/raw. "
                "Το dataset layout είναι ambiguous."
            )
        meta_file_path = path
        continue

    match = PAIR_PATTERN.match(file_name)

    if not match:
        auxiliary_csv_files.append(rel_path)
        continue

    file_type, park_id = match.groups()
    park_id = park_id.zfill(5)

    if file_type.lower() == "input":
        if park_id in input_files:
            duplicate_pair_rows.append(
                {
                    "park_id": park_id,
                    "file_type": "input",
                    "existing_path": to_raw_relative(input_files[park_id]),
                    "duplicate_path": rel_path,
                }
            )
        else:
            input_files[park_id] = path
    else:
        if park_id in target_files:
            duplicate_pair_rows.append(
                {
                    "park_id": park_id,
                    "file_type": "target",
                    "existing_path": to_raw_relative(target_files[park_id]),
                    "duplicate_path": rel_path,
                }
            )
        else:
            target_files[park_id] = path

if duplicate_pair_rows:
    duplicate_pair_df = pd.DataFrame(duplicate_pair_rows)
    display(duplicate_pair_df)

    raise RuntimeError(
        "Βρέθηκαν duplicate pair files για το ίδιο park_id. "
        "Το NB02 απαιτεί unambiguous one-input / one-target file mapping."
    )

if not input_files and not target_files:
    raise RuntimeError(
        "Δεν βρέθηκαν raw files που να ταιριάζουν στο expected pair pattern "
        "'data_input_<park_id>.csv' / 'data_target_<park_id>.csv'. "
        "Έλεγξε ότι τα αρχεία υπάρχουν κάτω από το data/raw/ ή data/raw/kassel_dataset/."
    )

paired_park_ids = sorted(set(input_files) & set(target_files))
missing_input_park_ids = sorted(set(target_files) - set(input_files))
missing_target_park_ids = sorted(set(input_files) - set(target_files))
all_park_ids = sorted(set(input_files) | set(target_files))

if not paired_park_ids:
    raise RuntimeError(
        "Δεν βρέθηκε κανένα πλήρες input-target pair. "
        "Το NB02 δεν μπορεί να συνεχίσει χωρίς τουλάχιστον ένα paired park."
    )

pair_audit_rows = []

for park_id in paired_park_ids:
    pair_audit_rows.append(
        {
            "park_id": park_id,
            "pair_status": "paired",
            "input_file": to_raw_relative(input_files[park_id]),
            "target_file": to_raw_relative(target_files[park_id]),
        }
    )

for park_id in missing_input_park_ids:
    pair_audit_rows.append(
        {
            "park_id": park_id,
            "pair_status": "missing_input",
            "input_file": None,
            "target_file": to_raw_relative(target_files[park_id]),
        }
    )

for park_id in missing_target_park_ids:
    pair_audit_rows.append(
        {
            "park_id": park_id,
            "pair_status": "missing_target",
            "input_file": to_raw_relative(input_files[park_id]),
            "target_file": None,
        }
    )

pair_audit_df = (
    pd.DataFrame(pair_audit_rows)
    .sort_values(["pair_status", "park_id"])
    .reset_index(drop=True)
)

print(f"Σύνολο CSV files κάτω από raw root : {len(all_csv_files)}")
print(f"Matched input files                : {len(input_files)}")
print(f"Matched target files               : {len(target_files)}")
print(f"Paired parks                       : {len(paired_park_ids)}")
print(f"Parks with missing input           : {len(missing_input_park_ids)}")
print(f"Parks with missing target          : {len(missing_target_park_ids)}")
print(f"Meta file found                    : {meta_file_path is not None}")

if meta_file_path is not None:
    print(f"Meta file path                     : {to_raw_relative(meta_file_path)}")

if auxiliary_csv_files:
    print(f"Auxiliary / non-pair CSV files     : {len(auxiliary_csv_files)}")
    display(pd.DataFrame({"auxiliary_csv_file": auxiliary_csv_files}).head(20))

display(pair_audit_df.head(20))

Σύνολο CSV files κάτω από raw root : 547
Matched input files                : 272
Matched target files               : 272
Paired parks                       : 272
Parks with missing input           : 0
Parks with missing target          : 0
Meta file found                    : True
Meta file path                     : kassel_dataset\meta.csv
Auxiliary / non-pair CSV files     : 2


,auxiliary_csv_file
0,ninja_evia_2024_raw.csv
1,wind_evia_2024_raw.csv


,park_id,pair_status,input_file,target_file
0,00011,paired,kassel_dataset\data_input_00011.csv,kassel_dataset\data_target_00011.csv
1,00090,paired,kassel_dataset\data_input_00090.csv,kassel_dataset\data_target_00090.csv
2,00096,paired,kassel_dataset\data_input_00096.csv,kassel_dataset\data_target_00096.csv
3,00161,paired,kassel_dataset\data_input_00161.csv,kassel_dataset\data_target_00161.csv
4,00164,paired,kassel_dataset\data_input_00164.csv,kassel_dataset\data_target_00164.csv
5,00183,paired,kassel_dataset\data_input_00183.csv,kassel_dataset\data_target_00183.csv
6,00197,paired,kassel_dataset\data_input_00197.csv,kassel_dataset\data_target_00197.csv
7,00198,paired,kassel_dataset\data_input_00198.csv,kassel_dataset\data_target_00198.csv
8,00232,paired,kassel_dataset\data_input_00232.csv,kassel_dataset\data_target_00232.csv
9,00282,paired,kassel_dataset\data_input_00282.csv,kassel_dataset\data_target_00282.csv


## Strict parsing policy

Με βάση τα raw samples που έχουν ελεγχθεί, το input-side timestamp representation **δεν είναι απολύτως ομοιογενές**.

Observed patterns στα input files:

- European slash format, π.χ. `8/12/2018 0:00`
- ISO-like format, π.χ. `2018-12-08 00:00:00`

Observed pattern στα target files:

- canonical ISO-like format, π.χ. `2018-12-08 00:00:00`

Άρα το strict policy είναι:

### Input-side allowed formats
- `%d/%m/%Y %H:%M`
- `%Y-%m-%d %H:%M:%S`

### Target-side allowed formats
- `%Y-%m-%d %H:%M:%S`

## Τι απαγορεύεται

Σε αυτό το notebook απαγορεύονται ρητά:

- `errors="coerce"` στο critical parsing path
- `infer_datetime_format`
- `format="mixed"`
- implicit `dayfirst=True` χωρίς explicit format declaration
- downstream reparsing από raw strings

Αν ένα αρχείο δεν ταιριάζει **ολόκληρο** σε κάποιο από τα δηλωμένα formats, αποτυγχάνει.

In [3]:
# ---------------------------------------------------------------------
# Strict separator και timestamp utilities
# ---------------------------------------------------------------------

# Επιτρεπτά input-side timestamp formats, βασισμένα στα observed raw samples.
INPUT_TS_FORMATS = [
    "%d/%m/%Y %H:%M",      # Παράδειγμα: 8/12/2018 0:00
    "%Y-%m-%d %H:%M:%S",   # Παράδειγμα: 2018-12-08 00:00:00
]

# Επιτρεπτό target-side timestamp format.
TARGET_TS_FORMATS = [
    "%Y-%m-%d %H:%M:%S",
]


def choose_separator(file_path: Path, file_type: str) -> tuple[pd.DataFrame, str]:
    """
    Επιλέγει separator με explicit candidate set.

    Γιατί όχι auto-sniffing:
    - θέλουμε deterministic συμπεριφορά
    - θέλουμε ελεγχόμενη scoring λογική
    - θέλουμε αποφυγή αδιαφανούς inference

    Extra robustness:
    - χειρίζεται καθαρά true zero-byte / EmptyDataError cases
    """
    separator_candidates = [";", ","]
    time_candidates = {"fcst_time", "time", "timestamp"}

    if file_type == "input":
        value_candidates = {"nwp_fcst_horiz_hours"}
        expected_sep = ";"
    else:
        value_candidates = {"pw", "icon_eu_daf_pc_baseline", "test_flag"}
        expected_sep = ","

    scored_reads = []
    read_errors = []

    for sep in separator_candidates:
        try:
            df = pd.read_csv(
                file_path,
                sep=sep,
                dtype="string",
                keep_default_na=False,
            )
        except pd.errors.EmptyDataError:
            read_errors.append((sep, "empty"))
            continue
        except Exception as exc:
            read_errors.append((sep, f"{type(exc).__name__}: {exc}"))
            continue

        df.columns = [str(col).strip() for col in df.columns]

        # Scoring logic:
        # - περισσότερες στήλες => πιο plausible parse
        # - ύπαρξη timestamp column => ισχυρή ένδειξη σωστού parse
        # - ύπαρξη βασικών value columns => επιπλέον ένδειξη
        # - μικρή προτίμηση στο observed expected separator
        score = 0
        score += len(df.columns)
        score += 50 if any(col in df.columns for col in time_candidates) else 0
        score += 10 if any(col in df.columns for col in value_candidates) else 0
        score += 1 if sep == expected_sep else 0

        # Αν η ανάγνωση κατέληξε σε μία μόνο στήλη, θεωρείται degenerate parse.
        if len(df.columns) == 1:
            score -= 100

        scored_reads.append((score, sep, df))

    if not scored_reads:
        if read_errors and all(msg == "empty" for _, msg in read_errors):
            raise ValueError(
                f"File is structurally empty: {file_path.name}"
            )

        raise ValueError(
            f"Separator detection failed for {file_path.name}. "
            f"Candidate errors={read_errors}"
        )

    scored_reads.sort(key=lambda x: x[0], reverse=True)
    _, best_sep, best_df = scored_reads[0]

    if len(best_df.columns) == 1:
        raise ValueError(
            f"Separator detection failed for {file_path.name}. "
            "All candidate parses remained degenerate."
        )

    return best_df, best_sep


def get_time_column(df: pd.DataFrame, file_type: str, file_name: str) -> str:
    """
    Εντοπίζει timestamp column με explicit file-type-specific search order.
    """
    if file_type == "input":
        candidates = ["fcst_time", "time", "timestamp"]
    else:
        candidates = ["time", "timestamp", "fcst_time"]

    for col in candidates:
        if col in df.columns:
            return col

    raise KeyError(
        f"No timestamp column found in {file_name}. "
        f"Available columns={df.columns.tolist()}"
    )


def parse_timestamp_strict(
    series: pd.Series,
    allowed_formats: list[str],
    field_name: str,
) -> tuple[pd.Series, str]:
    """
    Κάνει strict datetime parsing μόνο με τα δηλωμένα formats.

    Επιστρέφει:
    - parsed timestamp Series
    - το format που τελικά χρησιμοποιήθηκε

    Κρίσιμες αρχές:
    - όχι `errors="coerce"`
    - όχι mixed heuristics
    - αν ένα format δεν ταιριάζει σε όλη τη σειρά, απορρίπτεται
    """
    raw_str = series.astype("string").str.strip()
    empty_mask = raw_str.isna() | (raw_str == "")

    last_error = None

    for fmt in allowed_formats:
        try:
            parsed = pd.to_datetime(
                raw_str.mask(empty_mask),
                format=fmt,
                errors="raise",
            )
            return parsed, fmt
        except Exception as exc:
            last_error = exc

    sample_value = raw_str[~empty_mask].iloc[0] if (~empty_mask).any() else "<empty>"

    raise ValueError(
        f"Strict timestamp parsing failed for {field_name}. "
        f"Sample='{sample_value}' | Allowed formats={allowed_formats} | Last error={last_error}"
    )


def audit_time_series(ts: pd.Series) -> dict:
    """
    Παράγει συνοπτικό audit για το timestamp backbone πριν από canonical sorting.
    """
    return {
        "row_count": int(len(ts)),
        "nat_count": int(ts.isna().sum()),
        "duplicate_count": int(ts.duplicated().sum()),
        "monotonic_before_sort": bool(ts.is_monotonic_increasing),
        "min_time": ts.min(),
        "max_time": ts.max(),
    }


def validate_exact_timestamp_alignment(
    input_ts: pd.Series,
    target_ts: pd.Series,
) -> dict:
    """
    Απαιτεί exact one-to-one timestamp agreement μετά από canonical sorting.
    """
    input_key = pd.DataFrame(
        {"timestamp": input_ts.sort_values().reset_index(drop=True)}
    )
    target_key = pd.DataFrame(
        {"timestamp": target_ts.sort_values().reset_index(drop=True)}
    )

    outer_check = input_key.merge(
        target_key,
        on="timestamp",
        how="outer",
        indicator=True,
        validate="one_to_one",
    )

    left_only = int((outer_check["_merge"] == "left_only").sum())
    right_only = int((outer_check["_merge"] == "right_only").sum())
    both = int((outer_check["_merge"] == "both").sum())

    exact_match = (
        left_only == 0
        and right_only == 0
        and both == len(input_key) == len(target_key)
    )

    return {
        "left_only_count": left_only,
        "right_only_count": right_only,
        "matched_count": both,
        "exact_match": exact_match,
    }

In [4]:
# ---------------------------------------------------------------------
# Raw loading και strict pair validation
# ---------------------------------------------------------------------

def load_pair_raw(park_id: str) -> dict:
    """
    Φορτώνει ένα park pair σε raw μορφή και λύνει:
    - separator
    - timestamp column name
    """
    input_path = input_files[park_id]
    target_path = target_files[park_id]

    df_input, input_sep_used = choose_separator(input_path, file_type="input")
    df_target, target_sep_used = choose_separator(target_path, file_type="target")

    input_time_col = get_time_column(
        df_input,
        file_type="input",
        file_name=input_path.name,
    )
    target_time_col = get_time_column(
        df_target,
        file_type="target",
        file_name=target_path.name,
    )

    return {
        "park_id": park_id,
        "input_path": input_path,
        "target_path": target_path,
        "df_input": df_input,
        "df_target": df_target,
        "input_sep_used": input_sep_used,
        "target_sep_used": target_sep_used,
        "input_time_col": input_time_col,
        "target_time_col": target_time_col,
    }


def validate_loaded_pair(raw_pair: dict) -> dict:
    """
    Κάνει end-to-end strict validation ενός ήδη φορτωμένου pair.

    Validation sequence:
    1. Strict timestamp parsing
    2. Reject zero-row / structurally empty files
    3. Reject NaT timestamps
    4. Reject duplicate timestamps
    5. Audit raw ordering πριν από canonical sorting
    6. Canonical sort by timestamp
    7. Exact input-target timestamp equality
    8. One-to-one merge validity
    """
    df_input = raw_pair["df_input"].copy()
    df_target = raw_pair["df_target"].copy()

    input_time_col = raw_pair["input_time_col"]
    target_time_col = raw_pair["target_time_col"]

    # Strict parsing με explicit allowed formats.
    df_input[input_time_col], input_ts_format_used = parse_timestamp_strict(
        df_input[input_time_col],
        INPUT_TS_FORMATS,
        field_name=f"{raw_pair['input_path'].name}:{input_time_col}",
    )
    df_target[target_time_col], target_ts_format_used = parse_timestamp_strict(
        df_target[target_time_col],
        TARGET_TS_FORMATS,
        field_name=f"{raw_pair['target_path'].name}:{target_time_col}",
    )

    input_audit = audit_time_series(df_input[input_time_col])
    target_audit = audit_time_series(df_target[target_time_col])

    # Header-only ή zero-row αρχεία δεν επιτρέπονται να περάσουν downstream.
    if input_audit["row_count"] == 0:
        raise ValueError("Input file is structurally empty after decoding.")

    if target_audit["row_count"] == 0:
        raise ValueError("Target file is structurally empty after decoding.")

    # Οποιοδήποτε NaT μετά από strict parsing είναι σφάλμα.
    if input_audit["nat_count"] > 0:
        raise ValueError(
            f"Input contains {input_audit['nat_count']} NaT timestamps after strict parsing."
        )

    if target_audit["nat_count"] > 0:
        raise ValueError(
            f"Target contains {target_audit['nat_count']} NaT timestamps after strict parsing."
        )

    # Duplicate timestamps δεν επιτρέπονται πριν από το merge.
    if input_audit["duplicate_count"] > 0:
        raise ValueError(
            f"Input contains {input_audit['duplicate_count']} duplicate timestamps."
        )

    if target_audit["duplicate_count"] > 0:
        raise ValueError(
            f"Target contains {target_audit['duplicate_count']} duplicate timestamps."
        )

    # Canonical sorting πριν από οποιοδήποτε alignment check.
    df_input = df_input.sort_values(input_time_col).reset_index(drop=True)
    df_target = df_target.sort_values(target_time_col).reset_index(drop=True)

    alignment = validate_exact_timestamp_alignment(
        df_input[input_time_col],
        df_target[target_time_col],
    )

    if not alignment["exact_match"]:
        raise ValueError(
            "Input-target timestamp mismatch detected. "
            f"left_only={alignment['left_only_count']}, "
            f"right_only={alignment['right_only_count']}, "
            f"matched={alignment['matched_count']}"
        )

    # Το merge πρέπει να είναι αυστηρά one-to-one.
    merged_df = df_input.merge(
        df_target,
        left_on=input_time_col,
        right_on=target_time_col,
        how="inner",
        validate="one_to_one",
    )

    # Από εδώ και πέρα κρατάμε ένα μόνο canonical timestamp column.
    merged_df = merged_df.rename(columns={input_time_col: "timestamp"})

    if target_time_col != "timestamp" and target_time_col in merged_df.columns:
        merged_df = merged_df.drop(columns=[target_time_col])

    merged_df = merged_df.sort_values("timestamp").reset_index(drop=True)

    return {
        **raw_pair,
        "df_input": df_input,
        "df_target": df_target,
        "merged_df": merged_df,
        "input_ts_format_used": input_ts_format_used,
        "target_ts_format_used": target_ts_format_used,
        "input_audit": input_audit,
        "target_audit": target_audit,
        "alignment": alignment,
    }


def load_and_validate_pair(park_id: str) -> dict:
    """
    Convenience wrapper για πλήρες strict validation ενός park.
    """
    raw_pair = load_pair_raw(park_id)
    return validate_loaded_pair(raw_pair)

## Deterministic single-park validation

Πριν από το full audit, γίνεται deterministic sample validation.

Στόχος αυτού του βήματος δεν είναι να υποκαταστήσει το all-parks audit, αλλά να δείξει καθαρά σε ένα πραγματικό example:

- ποιο separator επιλέχθηκε
- ποια timestamp columns χρησιμοποιήθηκαν
- ποιο timestamp format ταίριαξε
- αν υπάρχουν duplicates ή NaT
- αν το raw ordering ήταν ήδη monotonic
- αν το exact timestamp alignment πέρασε
- αν το τελικό merge είναι one-to-one

Εφόσον υπάρχει, προτιμάμε ως deterministic sample το `park_id = 00011`, επειδή είναι γνωστό raw heterogeneity case.

In [5]:
# ---------------------------------------------------------------------
# Deterministic single-park validation
# ---------------------------------------------------------------------

preferred_sample_park_id = "00011"
sample_park_id = (
    preferred_sample_park_id
    if preferred_sample_park_id in paired_park_ids
    else paired_park_ids[0]
)

try:
    sample_result = load_and_validate_pair(sample_park_id)

    print(f"Sample park              : {sample_result['park_id']}")
    print(f"Input file               : {to_raw_relative(sample_result['input_path'])}")
    print(f"Target file              : {to_raw_relative(sample_result['target_path'])}")
    print(f"Input separator used     : {sample_result['input_sep_used']}")
    print(f"Target separator used    : {sample_result['target_sep_used']}")
    print(f"Input timestamp column   : {sample_result['input_time_col']}")
    print(f"Target timestamp column  : {sample_result['target_time_col']}")
    print(f"Input timestamp format   : {sample_result['input_ts_format_used']}")
    print(f"Target timestamp format  : {sample_result['target_ts_format_used']}")

    sample_summary_df = pd.DataFrame(
        [
            {
                "park_id": sample_result["park_id"],
                "input_rows": sample_result["input_audit"]["row_count"],
                "target_rows": sample_result["target_audit"]["row_count"],
                "merged_rows": len(sample_result["merged_df"]),
                "input_nat_count": sample_result["input_audit"]["nat_count"],
                "target_nat_count": sample_result["target_audit"]["nat_count"],
                "input_duplicate_count": sample_result["input_audit"]["duplicate_count"],
                "target_duplicate_count": sample_result["target_audit"]["duplicate_count"],
                "input_monotonic_before_sort": sample_result["input_audit"]["monotonic_before_sort"],
                "target_monotonic_before_sort": sample_result["target_audit"]["monotonic_before_sort"],
                "left_only_count": sample_result["alignment"]["left_only_count"],
                "right_only_count": sample_result["alignment"]["right_only_count"],
                "matched_count": sample_result["alignment"]["matched_count"],
                "exact_timestamp_match": sample_result["alignment"]["exact_match"],
                "input_min_time": sample_result["input_audit"]["min_time"],
                "input_max_time": sample_result["input_audit"]["max_time"],
                "target_min_time": sample_result["target_audit"]["min_time"],
                "target_max_time": sample_result["target_audit"]["max_time"],
            }
        ]
    )

    display(sample_summary_df)
    display(sample_result["merged_df"].head())

except Exception as exc:
    sample_result = None
    print(f"Deterministic sample validation failed for park {sample_park_id}.")
    print(f"Reason: {exc}")

Sample park              : 00011
Input file               : kassel_dataset\data_input_00011.csv
Target file              : kassel_dataset\data_target_00011.csv
Input separator used     : ;
Target separator used    : ,
Input timestamp column   : fcst_time
Target timestamp column  : time
Input timestamp format   : %d/%m/%Y %H:%M
Target timestamp format  : %Y-%m-%d %H:%M:%S


,park_id,input_rows,target_rows,merged_rows,input_nat_count,target_nat_count,input_duplicate_count,target_duplicate_count,input_monotonic_before_sort,target_monotonic_before_sort,left_only_count,right_only_count,matched_count,exact_timestamp_match,input_min_time,input_max_time,target_min_time,target_max_time
0,00011,12430,12430,12430,0,0,0,0,True,True,0,0,12430,True,2018-12-08,2020-06-01 23:00:00,2018-12-08,2020-06-01 23:00:00


,timestamp,nwp_fcst_horiz_hours,T_HAG_2_M,RELHUM_HAG_2_M,PS_SFC_0_M,U_GVL_58_HL,V_GVL_58_HL,U_GVL_60_HL,V_GVL_60_HL,ASWDIFDS_SFC_0_M,ASWDIRS_SFC_0_M,U_GVL_58_HL_m1,V_GVL_58_HL_m1,U_GVL_60_HL_m1,V_GVL_60_HL_m1,U_GVL_58_HL_p1,V_GVL_58_HL_p1,U_GVL_60_HL_p1,V_GVL_60_HL_p1,test_flag,pw,icon_eu_daf_pc_baseline
0,2018-12-08 00:00:00,24,277105,79144,93060656,11219,-1154,6280,-324,25250,15902,12240,-1231,6980,-473,10205,-2168,5744,-1034,0,0.100,0.806
1,2018-12-08 01:00:00,25,276581,80629,93124426,12240,-1231,6980,-473,24238,15266,11536,346,6644,336,11219,-1154,6280,-324,0,0.126,0.895
2,2018-12-08 02:00:00,26,276119,81829,93155008,11536,346,6644,336,23307,14680,12284,1659,7124,1062,12240,-1231,6980,-473,0,0.232,0.829
3,2018-12-08 03:00:00,27,275763,80457,93182648,12284,1659,7124,1062,22443,14137,12939,968,7530,627,11536,346,6644,336,0,0.248,0.901
4,2018-12-08 04:00:00,28,275410,79750,93234324,12939,968,7530,627,21643,13629,11216,2137,6384,1384,12284,1659,7124,1062,0,0.232,0.937


## Full strict raw audit

Το all-parks audit είναι το βασικό output του NB02.

Το audit artifact πρέπει να διακρίνει καθαρά:

- parks που πέρασαν πλήρως (`ok`)
- parks που πέρασαν με ordering warning (`warning`)
- parks που απέτυχαν (`failed`)
- parks με missing input / target files
- parks με `empty_data`
- parks με timestamp parsing failure
- parks με duplicate timestamps
- parks με merge validation failure

Η φιλοσοφία είναι **fail-fast ανά park αλλά όχι fail-stop για όλο το notebook**:
το notebook συνεχίζει για να καταγράψει όλο το population-level audit table.

In [6]:
# ---------------------------------------------------------------------
# Audit row schema και logging wrapper
# ---------------------------------------------------------------------

def build_base_audit_row(park_id: str) -> dict:
    """
    Δημιουργεί το canonical audit row schema για ένα park.
    """
    return {
        "park_id": park_id,
        "status": None,
        "failure_stage": None,
        "failure_reason": None,
        "input_file": to_raw_relative(input_files.get(park_id)),
        "target_file": to_raw_relative(target_files.get(park_id)),
        "input_sep_used": None,
        "target_sep_used": None,
        "input_time_col": None,
        "target_time_col": None,
        "input_ts_format_used": None,
        "target_ts_format_used": None,
        "input_rows": None,
        "target_rows": None,
        "merged_rows": None,
        "input_nat_count": None,
        "target_nat_count": None,
        "input_duplicate_count": None,
        "target_duplicate_count": None,
        "input_monotonic_before_sort": None,
        "target_monotonic_before_sort": None,
        "input_min_time": None,
        "input_max_time": None,
        "target_min_time": None,
        "target_max_time": None,
        "left_only_count": None,
        "right_only_count": None,
        "matched_count": None,
        "exact_timestamp_match": None,
    }


def audit_pair_with_logging(park_id: str) -> dict:
    """
    Εκτελεί strict audit για ένα park και επιστρέφει πάντα audit row,
    ανεξάρτητα από το αν το park πέρασε ή απέτυχε.
    """
    row = build_base_audit_row(park_id)

    # Πρώτα ελέγχουμε τις missing-pair περιπτώσεις.
    if park_id not in input_files:
        row["status"] = "failed"
        row["failure_stage"] = "pairing"
        row["failure_reason"] = "Missing input file for this park ID."
        return row

    if park_id not in target_files:
        row["status"] = "failed"
        row["failure_stage"] = "pairing"
        row["failure_reason"] = "Missing target file for this park ID."
        return row

    # Μετά ελέγχουμε decoding / schema.
    try:
        raw_pair = load_pair_raw(park_id)

        row["input_sep_used"] = raw_pair["input_sep_used"]
        row["target_sep_used"] = raw_pair["target_sep_used"]
        row["input_time_col"] = raw_pair["input_time_col"]
        row["target_time_col"] = raw_pair["target_time_col"]

    except Exception as exc:
        message = str(exc)
        row["status"] = "failed"

        if "Separator detection failed" in message:
            row["failure_stage"] = "separator"
        elif "timestamp column" in message.lower():
            row["failure_stage"] = "schema"
        elif "empty" in message.lower():
            row["failure_stage"] = "empty_data"
        else:
            row["failure_stage"] = "raw_loading"

        row["failure_reason"] = message
        return row

    # Τέλος γίνεται το strict validation.
    try:
        result = validate_loaded_pair(raw_pair)

        row.update(
            {
                "input_ts_format_used": result["input_ts_format_used"],
                "target_ts_format_used": result["target_ts_format_used"],
                "input_rows": result["input_audit"]["row_count"],
                "target_rows": result["target_audit"]["row_count"],
                "merged_rows": len(result["merged_df"]),
                "input_nat_count": result["input_audit"]["nat_count"],
                "target_nat_count": result["target_audit"]["nat_count"],
                "input_duplicate_count": result["input_audit"]["duplicate_count"],
                "target_duplicate_count": result["target_audit"]["duplicate_count"],
                "input_monotonic_before_sort": result["input_audit"]["monotonic_before_sort"],
                "target_monotonic_before_sort": result["target_audit"]["monotonic_before_sort"],
                "input_min_time": result["input_audit"]["min_time"],
                "input_max_time": result["input_audit"]["max_time"],
                "target_min_time": result["target_audit"]["min_time"],
                "target_max_time": result["target_audit"]["max_time"],
                "left_only_count": result["alignment"]["left_only_count"],
                "right_only_count": result["alignment"]["right_only_count"],
                "matched_count": result["alignment"]["matched_count"],
                "exact_timestamp_match": result["alignment"]["exact_match"],
            }
        )

        # Αν το raw ordering δεν ήταν monotonic αλλά όλα τα strict checks πέρασαν,
        # καταγράφεται ως warning και όχι ως failed.
        if (
            result["input_audit"]["monotonic_before_sort"]
            and result["target_audit"]["monotonic_before_sort"]
        ):
            row["status"] = "ok"
        else:
            row["status"] = "warning"
            row["failure_stage"] = "ordering"
            row["failure_reason"] = (
                "Raw rows were not fully monotonic before canonical sorting, "
                "but strict parsing and exact timestamp alignment succeeded."
            )

        return row

    except Exception as exc:
        message = str(exc)
        row["status"] = "failed"

        if "strict timestamp parsing failed" in message.lower() or "nat timestamps" in message.lower():
            row["failure_stage"] = "timestamp_parsing"
        elif "duplicate timestamps" in message.lower():
            row["failure_stage"] = "duplicate_timestamp"
        elif "structurally empty" in message.lower():
            row["failure_stage"] = "empty_data"
        elif "timestamp mismatch" in message.lower():
            row["failure_stage"] = "merge_validation"
        else:
            row["failure_stage"] = "validation"

        row["failure_reason"] = message
        return row


# ---------------------------------------------------------------------
# All-parks strict audit execution
# ---------------------------------------------------------------------
audit_rows = []

for park_id in tqdm(all_park_ids, desc="Strict raw audit"):
    audit_rows.append(audit_pair_with_logging(park_id))

audit_df = pd.DataFrame(audit_rows)

display(audit_df.head(20))

Strict raw audit:   0%|          | 0/272 [00:00<?, ?it/s]

,park_id,status,failure_stage,failure_reason,input_file,target_file,input_sep_used,target_sep_used,input_time_col,target_time_col,input_ts_format_used,target_ts_format_used,input_rows,target_rows,merged_rows,input_nat_count,target_nat_count,input_duplicate_count,target_duplicate_count,input_monotonic_before_sort,target_monotonic_before_sort,input_min_time,input_max_time,target_min_time,target_max_time,left_only_count,right_only_count,matched_count,exact_timestamp_match
0,00011,ok,NaN,NaN,kassel_dataset\data_input_00011.csv,kassel_dataset\data_target_00011.csv,;,",",fcst_time,time,%d/%m/%Y %H:%M,%Y-%m-%d %H:%M:%S,12430.0,12430.0,12430.0,0.0,0.0,0.0,0.0,True,True,2018-12-08 00:00:00,2020-06-01 23:00:00,2018-12-08 00:00:00,2020-06-01 23:00:00,0.0,0.0,12430.0,True
1,00090,ok,NaN,NaN,kassel_dataset\data_input_00090.csv,kassel_dataset\data_target_00090.csv,",",",",fcst_time,time,%Y-%m-%d %H:%M:%S,%Y-%m-%d %H:%M:%S,12822.0,12822.0,12822.0,0.0,0.0,0.0,0.0,True,True,2018-12-08 00:00:00,2020-06-01 23:00:00,2018-12-08 00:00:00,2020-06-01 23:00:00,0.0,0.0,12822.0,True
2,00096,ok,NaN,NaN,kassel_dataset\data_input_00096.csv,kassel_dataset\data_target_00096.csv,",",",",fcst_time,time,%Y-%m-%d %H:%M:%S,%Y-%m-%d %H:%M:%S,9922.0,9922.0,9922.0,0.0,0.0,0.0,0.0,True,True,2019-04-09 12:00:00,2020-06-01 23:00:00,2019-04-09 12:00:00,2020-06-01 23:00:00,0.0,0.0,9922.0,True
3,00161,ok,NaN,NaN,kassel_dataset\data_input_00161.csv,kassel_dataset\data_target_00161.csv,",",",",fcst_time,time,%Y-%m-%d %H:%M:%S,%Y-%m-%d %H:%M:%S,12679.0,12679.0,12679.0,0.0,0.0,0.0,0.0,True,True,2018-12-08 00:00:00,2020-06-01 23:00:00,2018-12-08 00:00:00,2020-06-01 23:00:00,0.0,0.0,12679.0,True
4,00164,ok,NaN,NaN,kassel_dataset\data_input_00164.csv,kassel_dataset\data_target_00164.csv,",",",",fcst_time,time,%Y-%m-%d %H:%M:%S,%Y-%m-%d %H:%M:%S,12817.0,12817.0,12817.0,0.0,0.0,0.0,0.0,True,True,2018-12-08 00:00:00,2020-06-01 23:00:00,2018-12-08 00:00:00,2020-06-01 23:00:00,0.0,0.0,12817.0,True
5,00183,ok,NaN,NaN,kassel_dataset\data_input_00183.csv,kassel_dataset\data_target_00183.csv,",",",",fcst_time,time,%Y-%m-%d %H:%M:%S,%Y-%m-%d %H:%M:%S,12840.0,12840.0,12840.0,0.0,0.0,0.0,0.0,True,True,2018-12-08 00:00:00,2020-06-01 23:00:00,2018-12-08 00:00:00,2020-06-01 23:00:00,0.0,0.0,12840.0,True
6,00197,ok,NaN,NaN,kassel_dataset\data_input_00197.csv,kassel_dataset\data_target_00197.csv,",",",",fcst_time,time,%Y-%m-%d %H:%M:%S,%Y-%m-%d %H:%M:%S,12570.0,12570.0,12570.0,0.0,0.0,0.0,0.0,True,True,2018-12-08 00:00:00,2020-06-01 23:00:00,2018-12-08 00:00:00,2020-06-01 23:00:00,0.0,0.0,12570.0,True
7,00198,ok,NaN,NaN,kassel_dataset\data_input_00198.csv,kassel_dataset\data_target_00198.csv,",",",",fcst_time,time,%Y-%m-%d %H:%M:%S,%Y-%m-%d %H:%M:%S,12840.0,12840.0,12840.0,0.0,0.0,0.0,0.0,True,True,2018-12-08 00:00:00,2020-06-01 23:00:00,2018-12-08 00:00:00,2020-06-01 23:00:00,0.0,0.0,12840.0,True
8,00232,ok,NaN,NaN,kassel_dataset\data_input_00232.csv,kassel_dataset\data_target_00232.csv,",",",",fcst_time,time,%Y-%m-%d %H:%M:%S,%Y-%m-%d %H:%M:%S,12828.0,12828.0,12828.0,0.0,0.0,0.0,0.0,True,True,2018-12-08 00:00:00,2020-06-01 23:00:00,2018-12-08 00:00:00,2020-06-01 23:00:00,0.0,0.0,12828.0,True
9,00282,ok,NaN,NaN,kassel_dataset\data_input_00282.csv,kassel_dataset\data_target_00282.csv,",",",",fcst_time,time,%Y-%m-%d %H:%M:%S,%Y-%m-%d %H:%M:%S,12836.0,12836.0,12836.0,0.0,0.0,0.0,0.0,True,True,2018-12-08 00:00:00,2020-06-01 23:00:00,2018-12-08 00:00:00,2020-06-01 23:00:00,0.0,0.0,12836.0,True


In [7]:
# ---------------------------------------------------------------------
# Audit summaries και exports
# ---------------------------------------------------------------------

status_counts = (
    audit_df["status"]
    .value_counts(dropna=False)
    .rename_axis("status")
    .reset_index(name="parks")
)

pair_status_counts = (
    pair_audit_df["pair_status"]
    .value_counts(dropna=False)
    .rename_axis("pair_status")
    .reset_index(name="parks")
)

ok_df = audit_df[audit_df["status"] == "ok"].copy()
warning_df = audit_df[audit_df["status"] == "warning"].copy()
failed_df = audit_df[audit_df["status"] == "failed"].copy()

print(f"OK parks      : {len(ok_df)}")
print(f"Warning parks : {len(warning_df)}")
print(f"Failed parks  : {len(failed_df)}")

display(status_counts)
display(pair_status_counts)

# Συνοπτική εικόνα για το separator behavior.
if not audit_df.empty:
    separator_summary_df = (
        audit_df.groupby(["input_sep_used", "target_sep_used"], dropna=False)
        .size()
        .reset_index(name="parks")
        .sort_values("parks", ascending=False)
        .reset_index(drop=True)
    )
else:
    separator_summary_df = pd.DataFrame()

print("Separator usage summary:")
display(separator_summary_df)

# Συνολικό validated timestamp window μόνο για parks που πέρασαν.
valid_df = audit_df[audit_df["status"].isin(["ok", "warning"])].copy()

if not valid_df.empty:
    print("Global validated timestamp range:")
    print("Input : ", valid_df["input_min_time"].min(), "->", valid_df["input_max_time"].max())
    print("Target: ", valid_df["target_min_time"].min(), "->", valid_df["target_max_time"].max())

# Εμφάνιση ordering warnings, αν υπάρχουν.
if not warning_df.empty:
    print("Ordering warnings:")
    display(
        warning_df[
            [
                "park_id",
                "input_monotonic_before_sort",
                "target_monotonic_before_sort",
                "failure_reason",
            ]
        ].head(20)
    )

# Εμφάνιση failed parks, αν υπάρχουν.
if not failed_df.empty:
    print("Failed parks:")
    display(
        failed_df[
            [
                "park_id",
                "failure_stage",
                "failure_reason",
                "input_file",
                "target_file",
            ]
        ].head(30)
    )

    failure_reason_summary_df = (
        failed_df.groupby(["failure_stage", "failure_reason"], dropna=False)
        .size()
        .reset_index(name="parks")
        .sort_values("parks", ascending=False)
        .reset_index(drop=True)
    )

    print("Top failure reasons:")
    display(failure_reason_summary_df.head(30))
else:
    failure_reason_summary_df = pd.DataFrame()

# ---------------------------------------------------------------------
# Canonical NB02 exports
# ---------------------------------------------------------------------
audit_path = PROCESSED_DIR / "nb02_strict_raw_audit.csv"
pair_audit_path = PROCESSED_DIR / "nb02_pair_audit.csv"
status_counts_path = PROCESSED_DIR / "nb02_status_counts.csv"
separator_summary_path = PROCESSED_DIR / "nb02_separator_summary.csv"
failure_reason_summary_path = PROCESSED_DIR / "nb02_failure_reason_summary.csv"

audit_df.to_csv(audit_path, index=False)
pair_audit_df.to_csv(pair_audit_path, index=False)
status_counts.to_csv(status_counts_path, index=False)
separator_summary_df.to_csv(separator_summary_path, index=False)
failure_reason_summary_df.to_csv(failure_reason_summary_path, index=False)

print("NB02 artifacts saved successfully:")
print(f" - {audit_path}")
print(f" - {pair_audit_path}")
print(f" - {status_counts_path}")
print(f" - {separator_summary_path}")
print(f" - {failure_reason_summary_path}")

OK parks      : 271
Warning parks : 0
Failed parks  : 1


,status,parks
0,ok,271
1,failed,1


,pair_status,parks
0,paired,272


Separator usage summary:


,input_sep_used,target_sep_used,parks
0,",",",",271
1,;,",",1


Global validated timestamp range:
Input :  2018-01-25 00:00:00 -> 2020-06-01 23:00:00
Target:  2018-01-25 00:00:00 -> 2020-06-01 23:00:00
Failed parks:


,park_id,failure_stage,failure_reason,input_file,target_file
224,06238,empty_data,Input file is structurally empty after decoding.,kassel_dataset\data_input_06238.csv,kassel_dataset\data_target_06238.csv


Top failure reasons:


,failure_stage,failure_reason,parks
0,empty_data,Input file is structurally empty after decoding.,1


NB02 artifacts saved successfully:
 - C:\Users\diony\Desktop\WindPower_DigitalTwin\data\processed\nb02_strict_raw_audit.csv
 - C:\Users\diony\Desktop\WindPower_DigitalTwin\data\processed\nb02_pair_audit.csv
 - C:\Users\diony\Desktop\WindPower_DigitalTwin\data\processed\nb02_status_counts.csv
 - C:\Users\diony\Desktop\WindPower_DigitalTwin\data\processed\nb02_separator_summary.csv
 - C:\Users\diony\Desktop\WindPower_DigitalTwin\data\processed\nb02_failure_reason_summary.csv


## Συμπέρασμα

Το `NB02` λειτουργεί πλέον ως το **canonical raw decoding / timestamp integrity notebook** της active pipeline.

Εξασφαλίζει ότι:

- το raw input-target pairing ελέγχεται αυστηρά
- το recursive raw discovery είναι συμβατό με nested dataset layout
- το `meta.csv` αναγνωρίζεται ως auxiliary dataset artifact
- το separator handling είναι explicit και reproducible
- το timestamp parsing είναι strict και declared
- δεν επιτρέπονται `NaT` timestamps
- δεν επιτρέπονται duplicate timestamps
- το input-target alignment είναι exact
- το merge είναι one-to-one
- τα structurally empty files αποτυγχάνουν ρητά
- το downstream pipeline δεν εξαρτάται από inferred ή reparsed timestamps

## Downstream contract προς NB03

Το `NB03` πρέπει να χρησιμοποιεί το `nb02_strict_raw_audit.csv` ως mandatory gate και να συνεχίζει μόνο με parks που έχουν status:

- `ok`
- `warning`

Δεν επιτρέπεται το `NB03` να ξανακάνει loose timestamp parsing από raw strings.

## Coverage audit και downstream eligibility contract

Το strict raw audit του `NB02` απαντά στο ερώτημα:

> «Το park pair είναι raw-valid;»

Όμως πριν από το `NB04` χρειάζεται και δεύτερο ερώτημα:

> «Το raw-valid park ανήκει πράγματι στο canonical downstream cohort;»

Αυτό το πρόσθετο βήμα **δεν ξανακάνει raw parsing** και **δεν αλλάζει** το canonical role του `NB02`.
Αντίθετα, επεκτείνει τα ήδη παραγμένα audit artifacts με:

- σύνδεση με το `meta.csv`
- έλεγχο train / test sample availability
- classification του coverage window
- explicit `nb04_eligible` flag

### Βασική αρχή
- `raw_valid` σημαίνει: πέρασε το strict raw validation (`status in {"ok", "warning"}`)
- `nb04_eligible` σημαίνει: είναι και raw-valid **και** ανήκει στο canonical πλήρες cohort για downstream master assembly / feature engineering

In [8]:
# ---------------------------------------------------------------------
# Meta merge και coverage-aware downstream audit
# ---------------------------------------------------------------------

# Το meta.csv είναι απαραίτητο για να περάσουμε από raw-valid logic
# σε downstream cohort logic.
if meta_file_path is None:
    raise RuntimeError(
        "Δεν βρέθηκε meta.csv στο raw root. "
        "Το coverage-aware audit δεν μπορεί να εκτελεστεί χωρίς metadata."
    )

meta_df = pd.read_csv(meta_file_path).copy()

# Κάνουμε canonical string park_id για ασφαλές merge.
meta_df["park_id"] = meta_df["loc_id"].astype(str).str.zfill(5)

coverage_audit_df = audit_df.copy()
coverage_audit_df["park_id"] = coverage_audit_df["park_id"].astype(str).str.zfill(5)

# Κρατάμε μόνο τα metadata columns που χρειάζονται downstream.
meta_keep_cols = [
    "park_id",
    "turbine",
    "hub_height_m",
    "rotor_diameter_m",
    "nominal_power_kW",
    "long",
    "lat",
    "num_train_samples",
    "num_test_samples",
]

coverage_audit_df = coverage_audit_df.merge(
    meta_df[meta_keep_cols],
    on="park_id",
    how="left",
    validate="one_to_one",
)

# Flag για να ξέρουμε αν βρέθηκε metadata row.
coverage_audit_df["meta_row_found"] = (
    coverage_audit_df["turbine"].notna()
    | coverage_audit_df["hub_height_m"].notna()
    | coverage_audit_df["nominal_power_kW"].notna()
)

# Numeric normalization για τα sample counts.
coverage_audit_df["num_train_samples"] = (
    pd.to_numeric(coverage_audit_df["num_train_samples"], errors="coerce")
    .fillna(0)
    .astype(int)
)

coverage_audit_df["num_test_samples"] = (
    pd.to_numeric(coverage_audit_df["num_test_samples"], errors="coerce")
    .fillna(0)
    .astype(int)
)

# ---------------------------------------------------------------------
# Datetime parsing πάνω στα ήδη exported audit fields
# ---------------------------------------------------------------------
# Εδώ επιτρέπεται το errors='coerce' επειδή ΔΕΝ κάνουμε canonical raw parsing.
# Αναλύουμε μόνο τα timestamps που παρήχθησαν ήδη από το strict audit.
time_cols = [
    "input_min_time",
    "input_max_time",
    "target_min_time",
    "target_max_time",
]

for col in time_cols:
    coverage_audit_df[col] = pd.to_datetime(
        coverage_audit_df[col],
        errors="coerce",
    )

# ---------------------------------------------------------------------
# Βασικά downstream flags
# ---------------------------------------------------------------------
coverage_audit_df["raw_valid"] = coverage_audit_df["status"].isin(["ok", "warning"])
coverage_audit_df["has_train_samples"] = coverage_audit_df["num_train_samples"] > 0
coverage_audit_df["has_test_samples"] = coverage_audit_df["num_test_samples"] > 0

# Το canonical πλήρες window δεν το hard-code-άρουμε.
# Το αντλούμε από το modal πλήρες pattern των raw-valid parks.
valid_for_window_df = coverage_audit_df[coverage_audit_df["raw_valid"]].copy()

if valid_for_window_df.empty:
    raise RuntimeError(
        "Δεν υπάρχουν raw-valid parks για να εξαχθεί canonical coverage window."
    )

standard_window_start = valid_for_window_df["input_min_time"].mode().iloc[0]
standard_window_end = valid_for_window_df["input_max_time"].mode().iloc[0]

coverage_audit_df["standard_window_start"] = standard_window_start
coverage_audit_df["standard_window_end"] = standard_window_end

coverage_audit_df["is_full_standard_window"] = (
    (coverage_audit_df["input_min_time"] == standard_window_start)
    & (coverage_audit_df["input_max_time"] == standard_window_end)
    & (coverage_audit_df["target_min_time"] == standard_window_start)
    & (coverage_audit_df["target_max_time"] == standard_window_end)
)

print("Canonical full coverage window inferred from raw-valid population:")
print(f" - start: {standard_window_start}")
print(f" - end  : {standard_window_end}")

Canonical full coverage window inferred from raw-valid population:
 - start: 2018-12-08 00:00:00
 - end  : 2020-06-01 23:00:00


In [9]:
# ---------------------------------------------------------------------
# Coverage classification, NB04 eligibility και exports
# ---------------------------------------------------------------------

def classify_coverage_class(row: pd.Series) -> str:
    """
    Ταξινομεί κάθε park σε coverage-aware downstream class.

    Σημαντικό:
    - Το classification αυτό ΔΕΝ αντικαθιστά το strict raw audit status.
    - Απλώς προσθέτει cohort semantics για NB03 / NB04.
    """
    if not row["raw_valid"]:
        return "failed_raw_validation"

    if not row["meta_row_found"]:
        return "raw_valid_missing_meta"

    if row["is_full_standard_window"]:
        return "full_standard_window"

    if row["has_train_samples"] and row["has_test_samples"]:
        return "partial_mixed_window"

    if row["has_train_samples"] and not row["has_test_samples"]:
        return "partial_or_nonstandard_train_only"

    if row["has_test_samples"] and not row["has_train_samples"]:
        return "partial_or_nonstandard_test_only"

    return "raw_valid_but_no_split_counts"


coverage_audit_df["coverage_class"] = coverage_audit_df.apply(
    classify_coverage_class,
    axis=1,
)

# Το NB03 μπορεί να δει όλο το raw-valid population για validated-only inspection.
coverage_audit_df["nb03_validated_only"] = coverage_audit_df["raw_valid"]

# Για downstream master assembly / feature engineering πριν το NB04
# κρατάμε μόνο το αυστηρά canonical πλήρες cohort.
coverage_audit_df["nb04_eligible"] = (
    coverage_audit_df["raw_valid"]
    & coverage_audit_df["exact_timestamp_match"].fillna(False).astype(bool)
    & coverage_audit_df["is_full_standard_window"]
    & coverage_audit_df["has_train_samples"]
    & coverage_audit_df["has_test_samples"]
)

# ---------------------------------------------------------------------
# Summaries
# ---------------------------------------------------------------------
coverage_class_summary_df = (
    coverage_audit_df["coverage_class"]
    .value_counts(dropna=False)
    .rename_axis("coverage_class")
    .reset_index(name="parks")
)

nb04_eligibility_summary_df = pd.DataFrame(
    {
        "nb04_eligible": [True, False],
        "parks": [
            int(coverage_audit_df["nb04_eligible"].sum()),
            int((~coverage_audit_df["nb04_eligible"]).sum()),
        ],
    }
)

raw_valid_but_not_nb04_df = coverage_audit_df[
    coverage_audit_df["raw_valid"] & (~coverage_audit_df["nb04_eligible"])
].copy()

print("Coverage class summary:")
display(coverage_class_summary_df)

print("NB04 eligibility summary:")
display(nb04_eligibility_summary_df)

if not raw_valid_but_not_nb04_df.empty:
    print("Raw-valid αλλά όχι NB04-eligible parks:")
    display(
        raw_valid_but_not_nb04_df[
            [
                "park_id",
                "coverage_class",
                "input_min_time",
                "input_max_time",
                "num_train_samples",
                "num_test_samples",
                "exact_timestamp_match",
            ]
        ]
        .sort_values(["coverage_class", "park_id"])
        .reset_index(drop=True)
    )

# ---------------------------------------------------------------------
# Canonical coverage-aware exports
# ---------------------------------------------------------------------
coverage_audit_path = PROCESSED_DIR / "nb02_meta_coverage_audit.csv"
coverage_class_summary_path = PROCESSED_DIR / "nb02_coverage_class_summary.csv"
nb04_eligibility_summary_path = PROCESSED_DIR / "nb02_nb04_eligibility_summary.csv"

coverage_audit_df.to_csv(coverage_audit_path, index=False)
coverage_class_summary_df.to_csv(coverage_class_summary_path, index=False)
nb04_eligibility_summary_df.to_csv(nb04_eligibility_summary_path, index=False)

print("Coverage-aware NB02 artifacts saved successfully:")
print(f" - {coverage_audit_path}")
print(f" - {coverage_class_summary_path}")
print(f" - {nb04_eligibility_summary_path}")

# ---------------------------------------------------------------------
# Αυστηρά πρακτικό sanity print για το downstream contract
# ---------------------------------------------------------------------
print("\nDownstream contract snapshot:")
print(f" - Raw-valid parks          : {int(coverage_audit_df['raw_valid'].sum())}")
print(f" - NB04-eligible parks      : {int(coverage_audit_df['nb04_eligible'].sum())}")
print(f" - Excluded from NB04 cohort: {int((coverage_audit_df['raw_valid'] & (~coverage_audit_df['nb04_eligible'])).sum())}")

Coverage class summary:


,coverage_class,parks
0,full_standard_window,256
1,partial_mixed_window,8
2,partial_or_nonstandard_train_only,5
3,partial_or_nonstandard_test_only,2
4,failed_raw_validation,1


NB04 eligibility summary:


,nb04_eligible,parks
0,True,256
1,False,16


Raw-valid αλλά όχι NB04-eligible parks:


,park_id,coverage_class,input_min_time,input_max_time,num_train_samples,num_test_samples,exact_timestamp_match
0,00096,partial_mixed_window,2019-04-09 12:00:00,2020-06-01 23:00:00,5627,4295,True
1,00656,partial_mixed_window,2018-12-08 00:00:00,2020-04-22 06:00:00,8510,3318,True
2,02115,partial_mixed_window,2019-11-15 08:00:00,2020-06-01 23:00:00,341,4207,True
3,03660,partial_mixed_window,2019-01-07 13:00:00,2020-06-01 23:00:00,7436,4295,True
4,03905,partial_mixed_window,2019-01-03 11:00:00,2020-06-01 23:00:00,6889,3898,True
5,04279,partial_mixed_window,2018-12-08 00:00:00,2020-04-23 08:00:00,8545,3344,True
6,04745,partial_mixed_window,2018-12-08 00:00:00,2020-05-31 23:00:00,8540,4259,True
7,06101,partial_mixed_window,2018-12-10 05:00:00,2020-06-01 23:00:00,8440,4289,True
8,15976,partial_or_nonstandard_test_only,2020-04-01 00:00:00,2020-06-01 23:00:00,0,1416,True
9,15978,partial_or_nonstandard_test_only,2020-04-01 00:00:00,2020-06-01 23:00:00,0,1416,True


Coverage-aware NB02 artifacts saved successfully:
 - C:\Users\diony\Desktop\WindPower_DigitalTwin\data\processed\nb02_meta_coverage_audit.csv
 - C:\Users\diony\Desktop\WindPower_DigitalTwin\data\processed\nb02_coverage_class_summary.csv
 - C:\Users\diony\Desktop\WindPower_DigitalTwin\data\processed\nb02_nb04_eligibility_summary.csv

Downstream contract snapshot:
 - Raw-valid parks          : 271
 - NB04-eligible parks      : 256
 - Excluded from NB04 cohort: 15


## Επικαιροποιημένο downstream contract

Το `NB02` παραμένει η **μοναδική canonical raw-validation authority** του pipeline.

Πλέον όμως παράγει δύο διαφορετικά επίπεδα απόφασης:

1. **Strict raw validation**
   - canonical artifact: `nb02_strict_raw_audit.csv`
   - ερώτημα: «Πέρασε το park το strict parsing / alignment audit;»

2. **Coverage-aware downstream eligibility**
   - canonical artifact: `nb02_meta_coverage_audit.csv`
   - ερώτημα: «Ανήκει το raw-valid park στο canonical downstream cohort;»

### Τι σημαίνει αυτό για τα επόμενα notebooks

- Το `NB03` μπορεί να χρησιμοποιήσει το `nb02_meta_coverage_audit.csv` για validated-only inspection,
  αλλά πρέπει να γνωρίζει ότι **raw-valid δεν σημαίνει απαραίτητα cohort-equivalent**.

- Για οποιοδήποτε **master assembly / feature engineering / benchmark-ready** downstream στάδιο πριν από το `NB04`,
  ο default αυστηρός κανόνας πρέπει να είναι:

> κράτα μόνο parks με `nb04_eligible == True`

Άρα το pipeline δεν αντιμετωπίζει πλέον όλα τα `ok` parks ως ισοδύναμα.